In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
raw_data = project_root / "data" / "raw" / "customer_support_tickets.csv"
db = project_root / "data" / "processed" / "customer_support.db"

### Запрос 11. Типы обращений с низким средним рейтингом (CTE)
Через CTE считает средний `customer_satisfaction_rating` для каждого типа обращения, 
потмо во внешнем запросе оставляет только типы со средним рейтингом ниже 3, 
отсортированные по возрастанию рейтинга.


In [2]:
q11 = '''
WITH type_ratings AS(
    SELECT 
        tt.ticket_type_name, 
        AVG(t.customer_satisfaction_rating) as avg_rating
    FROM tickets t
    JOIN ticket_types tt ON tt.ticket_type_id = t.ticket_type_id
    GROUP BY ticket_type_name
)
SELECT *
FROM type_ratings
WHERE avg_rating < 3
ORDER BY avg_rating;
'''
with sqlite3.connect(db) as conn:
    result11 = pd.read_sql(q11, conn)

result11

,ticket_type_name,avg_rating
0,Refund request,2.934564
1,Technical issue,2.958621


### Запрос 12. VIEW критичных тикетов
Создает представление `critical_tickets_view`, сохраняющее только тикеты 
с приоритетом Critical вместе с именем клиента, названием продукта и каналом обращения. 


In [3]:
create_view_q = '''
CREATE VIEW IF NOT EXISTS critical_tickets_view AS
SELECT 
    t.ticket_id,
    c.customer_name,
    p.product_name,
    t.ticket_channel,
    t.customer_satisfaction_rating
FROM tickets t
JOIN customers c ON c.customer_id = t.customer_id
JOIN products p ON p.product_id = t.product_id
WHERE t.ticket_priority = 'Critical';
'''

check_q = '''
SELECT * FROM critical_tickets_view
LIMIT 5;
'''

with sqlite3.connect(db) as conn:
    conn.execute(create_view_q)
    result12 = pd.read_sql(check_q, conn)

result12

,ticket_id,customer_name,product_name,ticket_channel,customer_satisfaction_rating
0,1,Marisa Obrien,GoPro Hero,Social media,None
1,2,Jessica Rios,LG Smart TV,Chat,None
2,7,Jacqueline Wright,Microsoft Surface,Social media,None
3,8,Denise Lee,Philips Hue Lights,Social media,None
4,10,William Dawson,Dyson Vacuum Cleaner,Phone,None


### Запрос 13. Порядковый номер тикета по продукту во времени
Для каждого продукта нумерует связанные с ним тикеты по дате покупки и 
показывает, каким по счёту был тикет во времени для конкретного продукта.

In [5]:
q13 = '''
SELECT
    t.ticket_id,
    t.product_id,
    t.purchase_date,
    ROW_NUMBER() OVER (
        PARTITION BY t.product_id
        ORDER BY t.purchase_date
    ) as purchase_order_num
    FROM tickets t;
'''

with sqlite3.connect(db) as conn:
    result13 = pd.read_sql(q13, conn)

result13

,ticket_id,product_id,purchase_date,purchase_order_num
0,4460,1,2020-01-02 00:00:00,1
1,7981,1,2020-01-03 00:00:00,2
2,5914,1,2020-01-06 00:00:00,3
3,3342,1,2020-01-07 00:00:00,4
4,6028,1,2020-01-07 00:00:00,5
...,...,...,...,...
8464,7598,42,2021-12-06 00:00:00,182
8465,7672,42,2021-12-14 00:00:00,183
8466,6405,42,2021-12-15 00:00:00,184
8467,2353,42,2021-12-22 00:00:00,185


### Запрос 14. Рейтинг предыдущего тикета клиента
Для каждого клиента, отсортированного по дате покупки, показывает рейтинг 
удовлетворенности из его предыдущего тикета - позволяет отследить динамику 
удовлетворенности одного клиента во времени


In [6]:
q14 = '''
SELECT
    t.customer_id,
    t.ticket_id,
    t.purchase_date,
    t.customer_satisfaction_rating,
    LAG(t.customer_satisfaction_rating) OVER (
        PARTITION BY t.customer_id
        ORDER BY t.purchase_date
    ) as previous_rating
    FROM tickets t;

'''

with sqlite3.connect(db) as conn:
    result14 = pd.read_sql(q14, conn)

result14

,customer_id,ticket_id,purchase_date,customer_satisfaction_rating,previous_rating
0,1,1,2021-03-22 00:00:00,NaN,NaN
1,2,2,2021-05-22 00:00:00,NaN,NaN
2,3,3,2020-07-14 00:00:00,3.0,NaN
3,4,4,2020-11-13 00:00:00,3.0,NaN
4,5,5,2020-02-04 00:00:00,1.0,NaN
...,...,...,...,...,...
8464,8316,8465,2021-12-08 00:00:00,NaN,NaN
8465,8317,8466,2020-02-22 00:00:00,NaN,NaN
8466,8318,8467,2021-08-17 00:00:00,3.0,NaN
8467,8319,8468,2021-10-16 00:00:00,3.0,NaN


### Запрос 15. Сравнение рейтинга тикета со средним по приоритету
Для каждого тикета показывает его рейтинг рядом со средним рейтингом 
по всем тикетам того же приоритета, без схлопывания строк


In [7]:
q15 = '''
SELECT 
    t.ticket_id,
    t.ticket_priority,
    t.customer_satisfaction_rating,
    AVG(t.customer_satisfaction_rating) OVER (
        PARTITION BY t.ticket_priority
    ) as avg_rating_for_priority

    FROM tickets t;
'''

with sqlite3.connect(db) as conn:
    result15 = pd.read_sql(q15, conn)

result15

,ticket_id,ticket_priority,customer_satisfaction_rating,avg_rating_for_priority
0,1,Critical,NaN,2.958678
1,2,Critical,NaN,2.958678
2,7,Critical,NaN,2.958678
3,8,Critical,NaN,2.958678
4,10,Critical,NaN,2.958678
...,...,...,...,...
8464,8449,Medium,2.0,2.976945
8465,8459,Medium,NaN,2.976945
8466,8462,Medium,NaN,2.976945
8467,8464,Medium,NaN,2.976945
